# Day 1 Demo — Notebook 3 of 3: Graph RAG — Fixing the Conflation

Same dataset and same failure case as notebook 2, now backed by an explicit
graph (Neo4j) instead of pure vector similarity. `connected_order: None` is a
structural, checkable fact — not a similarity judgment the LLM has to get
right on its own.

## Setup (Colab)

Add `OPENAI_API_KEY`, `NEO4J_URI`, `NEO4J_USER`, and `NEO4J_PASSWORD` to Colab's Secrets manager before running.

In [8]:
%pip install -q sentence-transformers openai python-dotenv neo4j

In [9]:
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

from openai import OpenAI
client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Dataset (same corpus as notebooks 1 & 2)

In [10]:
categories = {
    "Orders": [
        "Customer placed an order for laptops",
        "Client purchased new office chairs",
        "The account holder bought a batch of monitors",
        "Purchase order submitted for networking equipment",
        "New order received for printer cartridges",
        "Bulk order placed for server racks",
        "Customer requested a quote before ordering keyboards",
        "Order confirmation sent for the tablet shipment",
        "Client re-ordered the same set of webcams",
        "Purchase request approved for docking stations",
        "Customer added routers to their existing order",
        "Order placed for annual software licenses",
    ],
    "Billing": [
        "Invoice payment overdue by 45 days",
        "Billing statement shows a late payment fee",
        "Customer disputed a charge on their invoice",
        "Monthly billing cycle closed with no discrepancies",
        "Client requested a copy of their last invoice",
        "Payment reminder sent for outstanding balance",
        "Invoice was issued with an incorrect tax amount",
        "Customer set up autopay for future invoices",
        "Billing department flagged a duplicate charge",
        "Credit note issued against a cancelled invoice",
        "Client's payment bounced and needs reprocessing",
        "Invoice reconciliation completed for the quarter",
    ],
    "Shipping": [
        "Order shipped to customer's warehouse yesterday",
        "Product delivered to client's facility on time",
        "Shipment delayed due to a courier issue",
        "Package was returned to sender as undeliverable",
        "Tracking number issued for the outbound shipment",
        "Customer requested expedited delivery for their order",
        "Freight carrier confirmed pickup for the pallet",
        "Delivery attempted but no one was available",
        "Shipment split into two boxes due to weight limits",
        "Client's order arrived damaged in transit",
        "Logistics team rerouted the shipment through a new hub",
        "Customs held the international shipment for inspection",
    ],
    "Complaints": [
        "Customer support ticket opened for a defective laptop",
        "New complaint raised about a broken monitor",
        "Client reported the product stopped working after a week",
        "Support agent escalated the complaint to engineering",
        "Customer requested a replacement for the faulty unit",
        "Complaint closed after the refund was processed",
        "Client unhappy with the quality of the delivered item",
        "Support ticket reopened after the same issue recurred",
        "Customer threatened to cancel their contract over service quality",
        "Negative feedback submitted about installation support",
        "Client escalated their complaint to a account manager",
        "Repeated complaints logged about the same product line",
    ],
    "Contracts_Legal": [
        "Legal team reviewed the vendor contract renewal",
        "New service agreement signed with the client",
        "Contract clause disputed during the negotiation",
        "Non-disclosure agreement executed before the meeting",
        "Vendor contract terminated due to non-performance",
        "Legal flagged a liability clause in the draft agreement",
        "Master service agreement amended to include new terms",
        "Client requested an extension on the contract deadline",
        "Procurement finalized the terms of the supplier contract",
        "Contract renewal pending legal department sign-off",
        "Breach of contract notice sent to the vendor",
        "Service level agreement updated with new penalties",
    ],
    "HR_Payroll": [
        "Payroll processed for all employees this month",
        "New hire onboarding completed for the engineering team",
        "Employee submitted a request for annual leave",
        "HR flagged a discrepancy in the timesheet submission",
        "Performance review scheduled for next quarter",
        "Employee benefits enrollment window opened today",
        "Exit interview conducted for a departing team member",
        "Payroll correction issued for a missed overtime payment",
        "HR approved the internal transfer request",
        "New employee handbook distributed to all staff",
        "Recruitment team shortlisted candidates for the open role",
        "Employee raised a grievance about scheduling conflicts",
    ],
    "IT_Security": [
        "Security team detected unusual login activity",
        "IT support resolved a network outage in the office",
        "Password reset requested for a locked user account",
        "Firewall rules updated after the security audit",
        "Phishing attempt reported by an employee",
        "IT deployed a patch for the recent vulnerability",
        "Server downtime scheduled for maintenance overnight",
        "Access request approved for the new database",
        "Data backup completed successfully for all systems",
        "Security incident triggered an automatic account lockout",
        "IT ticket raised for a malfunctioning laptop",
        "VPN access restored after the configuration fix",
    ],
    "Sales_Leads": [
        "Sales rep followed up with a qualified lead",
        "New lead captured through the website contact form",
        "Prospect requested a product demo next week",
        "Sales pipeline updated with three new opportunities",
        "Lead converted into a paying customer",
        "Cold outreach campaign generated several new leads",
        "Sales team lost a deal to a competitor",
        "Discovery call scheduled with a potential enterprise client",
        "Lead marked as unqualified after initial screening",
        "Proposal sent to a prospect following their demo",
        "Sales forecast updated based on pipeline movement",
        "Referral lead came in from an existing customer",
    ],
    "Inventory": [
        "Warehouse stock count completed for the quarter",
        "Inventory levels low for a key product line",
        "Stock replenishment order triggered automatically",
        "Warehouse team flagged a discrepancy in stock records",
        "New shipment received and added to inventory",
        "Obsolete stock marked for clearance sale",
        "Inventory audit found several missing units",
        "Stock transferred between two regional warehouses",
        "Reorder point reached for the printer cartridge SKU",
        "Damaged inventory written off during the audit",
        "Warehouse capacity nearing its storage limit",
        "Cycle count scheduled for the electronics section",
    ],
    "Finance_Reports": [
        "Quarterly revenue report prepared for the board",
        "Finance team closed the books for the fiscal year",
        "Budget variance analysis completed for the department",
        "Expense report submitted for the recent business trip",
        "Finance flagged an unexpected increase in operating costs",
        "Annual financial statements sent for external audit",
        "Cash flow forecast updated for the next two quarters",
        "Cost center allocation reviewed by the finance team",
        "Profit margin analysis shared with senior leadership",
        "Finance approved the capital expenditure request",
        "Revenue recognition policy updated per new guidelines",
        "Departmental spending reconciled against the annual budget",
    ],
}

varied_texts = []
varied_labels = []
for category, sentences in categories.items():
    for s in sentences:
        varied_texts.append(s)
        varied_labels.append(category)

print(f"Total records: {len(varied_texts)}")
print(f"Categories: {len(categories)}")

Total records: 120
Categories: 10


## Connect to Neo4j Aura
**What Neo4j is**: a graph database — instead of storing data in rows/tables (like SQL) or as a bag of vectors (like the embedding indexes in notebook 1), it stores data as nodes (things — a Customer, an Order, a Warehouse) and relationships between them (PLACED_BY, RESTOCKS), each an explicit, first-class fact in the database. Querying it (with Cypher, its query language) means literally walking those connections — "does a path exist from this node to that one?" — rather than computing a similarity score.

Vector search tells you what's similar. A graph tells you what's actually connected. Those are different questions, and notebook 2's failure was mistaking the first for the second

In [7]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def test_connection():
    with driver.session() as session:
        result = session.run("RETURN 'Connection successful' AS message")
        return result.single()["message"]

print(test_connection())

Connection successful


## Build the graph: two intentionally disconnected clusters

The Orders/Shipping chain (outbound, customer-facing) and the Inventory/Warehouse chain (inbound, internal restock) — genuinely unrelated, sharing only a product name.

In [11]:
def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

# Clear any previous test data (safe on a fresh instance)
run_write("MATCH (n) DETACH DELETE n")

# Orders/Shipping chain
run_write("""
    CREATE (c:Customer {name: 'Acme Corp'})
    CREATE (o:Order {id: 'ORD-101', product: 'Printer cartridges', status: 'Fulfilled'})
    CREATE (s:Shipment {id: 'SHP-501', destination: "customer's warehouse", date: 'yesterday'})
    CREATE (o)-[:PLACED_BY]->(c)
    CREATE (o)-[:FULFILLED_BY]->(s)
""")

# Inventory/Warehouse chain — UNRELATED to the order above
run_write("""
    CREATE (w:Warehouse {name: 'Bengaluru DC'})
    CREATE (p:Product {name: 'Printer cartridges'})
    CREATE (r:InventoryReplenishment {id: 'REPL-77', date: 'today'})
    CREATE (r)-[:RESTOCKS]->(p)
    CREATE (r)-[:AT_WAREHOUSE]->(w)
""")

print("Graph created")

Graph created


Notice these two chains share nothing — no edge, no shared node — even though both mention 'Printer cartridges.' That's deliberate. They're two separate real-world events that happen to use the same product name, and the whole point of notebook 3 is proving the graph knows they're unrelated, where the embedding-based version in notebook 2 didn't.

## Verify: is the replenishment connected to any order?

Asks specifically about the inventory side — what the notebook 2 question ("recent order for stock replenishment") was actually asking about.

In [12]:
def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

result = run_read("""
    MATCH (r:InventoryReplenishment)-[:RESTOCKS]->(p:Product)
    OPTIONAL MATCH (r)-[:AT_WAREHOUSE]->(w:Warehouse)
    RETURN r.id AS replenishment_id, p.name AS product, w.name AS warehouse, r.date AS date
""")
print("Inventory replenishment records:")
for row in result:
    print(row)

result2 = run_read("""
    MATCH (r:InventoryReplenishment)
    OPTIONAL MATCH (r)-[*]-(o:Order)
    RETURN r.id AS replenishment_id, o.id AS connected_order
""")
print("\nIs the replenishment connected to any order?")
for row in result2:
    print(row)

Inventory replenishment records:
{'replenishment_id': 'REPL-77', 'product': 'Printer cartridges', 'warehouse': 'Bengaluru DC', 'date': 'today'}

Is the replenishment connected to any order?
{'replenishment_id': 'REPL-77', 'connected_order': None}


## Troubleshooting note: reconnecting after a stale driver

Aura Free can drop an idle connection (`ServiceUnavailable`). If a query fails after a pause, close and recreate the driver before retrying — don't assume the instance itself is down.

**Variable name note:** this cell deliberately uses `test_result` instead of `result` for its sanity-check query. `result` is already holding your real query output from the cell above (the list of dicts `run_read()` returned) — naming this one `result` too would silently overwrite it with a raw, non-subscriptable Neo4j `Result` object. If you ever add your own quick test query here, give it its own variable name rather than reusing `result`.

In [ ]:
driver.close()
driver = GraphDatabase.driver(uri, auth=(user, password))

with driver.session() as session:
    test_result = session.run("RETURN 'reconnected' AS status")
    print(test_result.single()["status"])

## From query results to LLM context — generically, not hardcoded

The earlier version of this cell hand-wrote a sentence per field (`replenishment_id`, `product`, `warehouse`, `date`) — that only works because this one demo query happens to have exactly those fields. A different query would need a different hardcoded sentence, which doesn't scale past a handful of demo queries.

`facts_from_rows()` below instead works on **any** list of row-dicts, whatever fields they contain — it doesn't know or care what a "replenishment" or a "warehouse" is. This is the same pattern a production system uses: whatever Cypher query ran, however many fields it returned, this turns it into plain-text facts the LLM can read, without a human writing a new template for every possible query shape.

In [13]:
def facts_from_rows(rows, label="Fact"):
    """Turns a list of row-dicts (from run_read()) into plain-text fact lines,
    generically — no field names are hardcoded, so this works for any query."""
    lines = []
    for row in rows:
        line = "; ".join(f"{k}: {v}" for k, v in row.items())
        lines.append(f"{label}: {line}")
    return "\n".join(lines)

# Run it on both query results from the cell above and print what it produces —
# this is exactly the text that gets handed to the LLM next, nothing hidden.
serialized_facts = facts_from_rows(result) + "\n" + facts_from_rows(result2)
print("Generic serialization output:\n")
print(serialized_facts)

Generic serialization output:

Fact: replenishment_id: REPL-77; product: Printer cartridges; warehouse: Bengaluru DC; date: today
Fact: replenishment_id: REPL-77; connected_order: None


## Generate the answer from clean graph facts

Same question as notebook 2's failure case — now grounded in the graph's explicit, *generically serialized* facts instead of raw similarity or a hand-written sentence.

In [14]:
prompt = f"""Using only the facts below, answer the question. Do not imply any connection between facts unless explicitly stated. If a fact's 'connected_order' is None, that means no connection exists — say so explicitly.

Facts:
{serialized_facts}

Question: Tell me about the recent order for stock replenishment.

Answer:"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)
print(response.choices[0].message.content)

The recent order for stock replenishment has the following details: replenishment_id is REPL-77, the product is Printer cartridges, it is located in the Bengaluru DC warehouse, and the date of the replenishment is today. There is no connected order associated with this replenishment_id.
